# Ollama Cloud Model Call Example
This notebook demonstrates how to call Ollama models via cloud API using LangChain.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get Ollama API credentials
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY")
OLLAMA_HOST = os.getenv("OLLAMA_HOST")

print(f"Ollama Host: {OLLAMA_HOST}")
print(f"API Key loaded: {bool(OLLAMA_API_KEY)}")

Ollama Host: https://ollama.com
API Key loaded: True


## Method 1: Direct API Call using requests

In [2]:
import requests
import json

# Ollama API endpoint for generating completions
url = f"{OLLAMA_HOST}/api/generate"

# Prepare the request payload
payload = {
    "model": "gpt-oss:20b",  # Replace with your model name if different
    "prompt": "What is machine learning?",
    "stream": False
}

# Add Authorization header
headers = {
    "Authorization": f"Bearer {OLLAMA_API_KEY}",
    "Content-Type": "application/json"
}

try:
    # Make the API call
    response = requests.post(url, json=payload, headers=headers, timeout=30)
    response.raise_for_status()  # Raise error for bad status codes
    
    # Parse and display the response
    result = response.json()
    print("Response from Ollama:")
    print(result.get("response", "No response"))
    
except requests.exceptions.RequestException as e:
    print(f"Error calling Ollama API: {e}")
    if hasattr(e.response, 'text'):
        print(f"Response text: {e.response.text}")

Response from Ollama:
**Machine learning (ML)** is a sub‑field of artificial intelligence (AI) that gives computers the ability to learn patterns and make decisions or predictions from data, without being explicitly programmed to perform each specific task.

---

## 1. Core Idea
- **Learning from data**: Instead of hard‑coding rules, an ML algorithm discovers relationships by observing examples.
- **Generalization**: After training on a set of data, the model should handle unseen data (new inputs) reasonably well.

---

## 2. Basic Components

| Element | What it is | Why it matters |
|---------|-----------|----------------|
| **Data** | Inputs and outputs (labels or signals) that the algorithm will use. | The foundation; ML quality hinges on data quality and quantity. |
| **Model** | A mathematical function (e.g., linear regression, neural network) that maps inputs to outputs. | Encodes the hypothesis that the algorithm will learn. |
| **Training** | Adjusting the model’s parameters t

### First, let's list available models

In [3]:
# List available models on the Ollama server
list_url = f"{OLLAMA_HOST}/api/tags"

headers = {
    "Authorization": f"Bearer {OLLAMA_API_KEY}",
    "Content-Type": "application/json"
}

try:
    response = requests.get(list_url, headers=headers, timeout=10)
    response.raise_for_status()
    
    models_data = response.json()
    print("Available models:")
    print(json.dumps(models_data, indent=2))
    
    if models_data.get("models"):
        available_models = [m.get("name") for m in models_data["models"]]
        print(f"\nModel names: {available_models}")
    
except requests.exceptions.RequestException as e:
    print(f"Error listing models: {e}")
    if hasattr(e, 'response') and hasattr(e.response, 'text'):
        print(f"Response: {e.response.text}")

Available models:
{
  "models": [
    {
      "name": "cogito-2.1:671b",
      "model": "cogito-2.1:671b",
      "modified_at": "2025-11-19T00:00:00Z",
      "size": 688586727753,
      "digest": "5c1168f3a867",
      "details": {
        "parent_model": "",
        "format": "",
        "family": "",
        "families": null,
        "parameter_size": "",
        "quantization_level": ""
      }
    },
    {
      "name": "glm-4.6",
      "model": "glm-4.6",
      "modified_at": "2025-09-29T00:00:00Z",
      "size": 696060000000,
      "digest": "ee0873722cc3",
      "details": {
        "parent_model": "",
        "format": "",
        "family": "",
        "families": null,
        "parameter_size": "",
        "quantization_level": ""
      }
    },
    {
      "name": "kimi-k2:1t",
      "model": "kimi-k2:1t",
      "modified_at": "2025-09-05T00:00:00Z",
      "size": 1118481408000,
      "digest": "7bb8dfabfd9c",
      "details": {
        "parent_model": "",
        "format": ""

## Method 2: Using LangChain with Ollama Cloud

In [5]:
# Using LangChain with Ollama
from langchain_community.llms import Ollama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Initialize Ollama LLM with cloud endpoint and API key
# Note: You may need to create a custom Ollama wrapper if the standard one doesn't support auth headers
try:
    llm = Ollama(
        model="gpt-oss:20b",
        base_url=OLLAMA_HOST,
        # Some versions allow passing custom headers
    )
    
    # Test simple invocation
    response = llm.invoke("What is artificial intelligence?")
    print("LangChain Ollama Response:")
    print(response)
    
except Exception as e:
    print(f"Error with LangChain Ollama: {e}")
    print("You may need to use Method 1 (direct API calls) instead")

C:\Users\achin\AppData\Local\Temp\ipykernel_13056\1625496190.py:9: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(


Error with LangChain Ollama: Ollama call failed with status code 401. Details: unauthorized

You may need to use Method 1 (direct API calls) instead


## Method 3: Custom Ollama Wrapper with Authentication

In [ ]:
from langchain_core.language_models import LLM
from langchain_core.callbacks.manager import CallbackManagerForLLMRun
from typing import Optional, List, Any
import requests

class OllamaCloudLLM(LLM):
    """Custom LLM wrapper for Ollama cloud API with authentication"""
    
    model_name: str = "gpt-oss"
    base_url: str = "https://ollama.com"
    api_key: str = ""
    
    @property
    def _llm_type(self) -> str:
        return "ollama_cloud"
    
    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> str:
        """Call the Ollama API with authentication"""
        
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        
        payload = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": False
        }
        
        url = f"{self.base_url}/api/generate"
        
        response = requests.post(url, json=payload, headers=headers, timeout=60)
        response.raise_for_status()
        
        result = response.json()
        return result.get("response", "")

# Initialize custom Ollama cloud LLM
ollama_cloud_llm = OllamaCloudLLM(
    model_name="gpt-oss",
    base_url=OLLAMA_HOST,
    api_key=OLLAMA_API_KEY
)

# Test the custom LLM
try:
    response = ollama_cloud_llm("Explain what is a transformer model in deep learning.")
    print("Custom Ollama Cloud LLM Response:")
    print(response)
except Exception as e:
    print(f"Error: {e}")